# Source experiment

Adapted from the submitted repository, commit `fce9ade29ef6de731407a7fcf14098f30a272cd3`, for the public-data workflow. Current definitions and figure mappings are documented in `docs/`.


In [ ]:
import os
from pathlib import Path
from IPython.display import display
_candidates = [Path.cwd(), *Path.cwd().parents]
_project = next((p for p in _candidates if (p / "src/mitoxy/data.py").is_file()), None)
if _project is None:
    raise RuntimeError("Launch this notebook within the Mitoxy repository")
os.environ["MITOXY_PROJECT_ROOT"] = str(_project)


# Figure 7 — public-data source-experiment analysis

This notebook reconstructs Figure 7 directly from the public GSE235046
primary-BMDM RNA-seq count matrix acquired by
`00_Paper_Data_Download.ipynb`.

The analysis performs positive-offset median-ratio normalization, PCA,
prespecified Equal4/Core3/SLC7A11 scoring, factorial interaction tests,
Torin-reversal tests, and a seeded 10,000-module expression-matched null.
Plot-ready tables are written to `data/processed/figure_7_clean`.

Quality control is limited to public-source integrity, experimental-design
structure, required variables, numerical validity, and output-file hashes.
No previously reported effect estimate or significance value is used to
accept or reject a result.

In [ ]:
from pathlib import Path
import hashlib
import os

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.decomposition import PCA


PROJECT_ROOT = Path(
    os.environ["MITOXY_PROJECT_ROOT"]
).resolve()
SOURCE_MANIFEST_PATH = (
    PROJECT_ROOT / "data" / "provenance" /
    "paper_public_source_manifest.csv"
)
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "figure_7_clean"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


source_manifest = pd.read_csv(SOURCE_MANIFEST_PATH)
source_row = source_manifest.loc[
    source_manifest["dataset"].eq("GSE235046")
    & source_manifest["role"].eq("BMDM count matrix")
]
if len(source_row) != 1:
    raise RuntimeError(
        "The paper source manifest must contain one GSE235046 count matrix"
    )
source_row = source_row.iloc[0]
COUNT_PATH = PROJECT_ROOT / str(source_row["relative_path"])
if not COUNT_PATH.is_file():
    raise FileNotFoundError(COUNT_PATH)
if COUNT_PATH.stat().st_size != int(source_row["size_bytes"]):
    raise RuntimeError("GSE235046 source-size mismatch")
if sha256_file(COUNT_PATH) != str(source_row["sha256"]):
    raise RuntimeError("GSE235046 source SHA-256 mismatch")

print("Count matrix:", COUNT_PATH)
print("Size (bytes):", COUNT_PATH.stat().st_size)
print("SHA-256:", sha256_file(COUNT_PATH))
print("Public-source integrity audit: PASS")

## Experimental design and count-matrix reconstruction

The deposited count columns use sequencing identifiers rather than GEO
titles. The explicit mapping below records the public GEO sample order and
experimental annotations in the analysis itself.

In [ ]:
sample_manifest = pd.DataFrame([
    ("GSM7493746", "BMDM_Media_01", "2544278_YW001", "Media", 1, 0, 0, 0, 9),
    ("GSM7493747", "BMDM_Media_02", "2544279_YW002", "Media", 2, 0, 0, 0, 9),
    ("GSM7493748", "BMDM_Media_03", "2544280_YW003", "Media", 3, 0, 0, 0, 9),
    ("GSM7493749", "BMDM_LPS_01", "2544281_YW004", "LPS", 1, 1, 0, 0, 9),
    ("GSM7493750", "BMDM_LPS_02", "2544282_YW005", "LPS", 2, 1, 0, 0, 9),
    ("GSM7493751", "BMDM_LPS_03", "2544283_YW006", "LPS", 3, 1, 0, 0, 9),
    ("GSM7493752", "BMDM_CS_01", "2544284_YW007", "CS", 1, 0, 1, 0, 9),
    ("GSM7493753", "BMDM_CS_02", "2544285_YW008", "CS", 2, 0, 1, 0, 9),
    ("GSM7493754", "BMDM_CS_03", "2544286_YW009", "CS", 3, 0, 1, 0, 9),
    ("GSM7493755", "BMDM_LPS+CS_01", "2544287_YW010", "LPS+CS", 1, 1, 1, 0, 9),
    ("GSM7493756", "BMDM_LPS+CS_02", "2544288_YW011", "LPS+CS", 2, 1, 1, 0, 9),
    ("GSM7493757", "BMDM_LPS+CS_03", "2544289_YW012", "LPS+CS", 3, 1, 1, 0, 9),
    ("GSM7493758", "BMDM_LPS+CS+Torin_01", "2544293_YW016", "LPS+CS+Torin", 1, 1, 1, 1, 9),
    ("GSM7493759", "BMDM_LPS+CS+Torin_02", "2544294_YW017", "LPS+CS+Torin", 2, 1, 1, 1, 9),
    ("GSM7493760", "BMDM_LPS+CS+Torin_03", "2544295_YW018", "LPS+CS+Torin", 3, 1, 1, 1, 9),
], columns=[
    "geo_accession", "sample_id", "count_column", "condition",
    "replicate", "lps", "carbon_starvation", "torin",
    "duration_hours",
])

CONDITIONS = ["Media", "LPS", "CS", "LPS+CS", "LPS+CS+Torin"]
TARGETS = ["BAK1", "BAX", "BID", "SLC7A11"]

if sample_manifest["sample_id"].duplicated().any():
    raise ValueError("Duplicated sample titles in the experimental design")
if sample_manifest["count_column"].duplicated().any():
    raise ValueError("Duplicated count columns in the experimental design")
design_counts = sample_manifest["condition"].value_counts()
if set(design_counts.index) != set(CONDITIONS) or not design_counts.eq(3).all():
    raise ValueError("GSE235046 must contain three samples in each condition")

raw = pd.read_csv(
    COUNT_PATH, sep="\t", compression="gzip", comment="#"
)
print("Raw count-table shape:", raw.shape)

required_count_columns = sample_manifest["count_column"].tolist()
absent_columns = sorted(set(required_count_columns) - set(raw.columns))
if absent_columns:
    raise KeyError(f"Count matrix is missing sample columns: {absent_columns}")

gene_candidates = [
    column for column in raw.columns
    if str(column).strip().casefold() in {
        "gene", "genes", "symbol", "genesymbol", "gene symbol",
        "genename", "gene name",
    }
]
if len(gene_candidates) != 1:
    raise RuntimeError(
        f"Could not uniquely identify the gene-symbol column: {gene_candidates}"
    )
gene_column = gene_candidates[0]

counts = raw[[gene_column] + required_count_columns].copy()
counts[gene_column] = counts[gene_column].astype(str).str.strip().str.upper()
for column in required_count_columns:
    counts[column] = pd.to_numeric(counts[column], errors="coerce")
if counts[required_count_columns].isna().any().any():
    raise ValueError("Non-numeric values were found in the count matrix")
if counts[required_count_columns].lt(0).any().any():
    raise ValueError("Negative values were found in the count matrix")

counts = (
    counts.dropna(subset=[gene_column])
    .groupby(gene_column, sort=False)[required_count_columns]
    .sum(min_count=1)
)
column_to_sample = sample_manifest.set_index("count_column")["sample_id"]
counts.columns = [column_to_sample[column] for column in counts.columns]
counts = counts[sample_manifest["sample_id"]]

missing_targets = sorted(set(TARGETS) - set(counts.index))
if missing_targets:
    raise KeyError(f"Target genes absent from count table: {missing_targets}")

display(sample_manifest)
display(counts.loc[TARGETS])
print("Genes before filtering:", len(counts))

## Positive-offset median-ratio normalization

In [ ]:
keep = (counts.ge(5).sum(axis=1) >= 3) & (counts.sum(axis=1) >= 20)
filtered = counts.loc[keep].astype(float)

# The +1 offset defines geometric means for genes with occasional zeros.
# Sample-wise numerators remain the observed counts.
geometric_means = np.exp(np.log(filtered + 1).mean(axis=1))
ratios = filtered.div(geometric_means, axis=0)
size_factors = ratios.median(axis=0)
if not np.isfinite(size_factors).all() or not size_factors.gt(0).all():
    raise ValueError("Invalid median-ratio size factors")
size_factors = size_factors / np.exp(np.log(size_factors).mean())

normalized = filtered.div(size_factors, axis=1)
log_expression = np.log2(normalized + 0.5)
if len(log_expression) <= 5000:
    raise ValueError("Too few genes remain after expression filtering")
if not np.isfinite(log_expression.to_numpy()).all():
    raise ValueError("Normalized expression contains non-finite values")
if not set(TARGETS).issubset(log_expression.index):
    raise KeyError("A target gene was removed by expression filtering")

normalization_audit = pd.DataFrame({
    "sample_id": size_factors.index,
    "size_factor": size_factors.values,
}).merge(
    sample_manifest, on="sample_id", how="left", validate="one_to_one"
)
normalization_audit.to_csv(
    OUTPUT_DIR / "normalization_audit.csv", index=False
)
print("Retained genes:", len(log_expression))
display(normalization_audit)

## Panel B — principal-component analysis

In [ ]:
variances = log_expression.var(axis=1, ddof=1)
top_genes = variances.nlargest(5000).index
pca = PCA(n_components=2, svd_solver="full")
pca_values = pca.fit_transform(log_expression.loc[top_genes].T)

panel_b = pd.DataFrame({
    "sample_id": log_expression.columns,
    "pc1": pca_values[:, 0],
    "pc2": pca_values[:, 1],
}).merge(
    sample_manifest, on="sample_id", how="left", validate="one_to_one"
)
panel_b["pc1_variance_pct"] = pca.explained_variance_ratio_[0] * 100
panel_b["pc2_variance_pct"] = pca.explained_variance_ratio_[1] * 100

if panel_b["condition"].isna().any():
    raise ValueError("PCA samples did not map to experimental conditions")
if not np.isfinite(panel_b[[
    "pc1", "pc2", "pc1_variance_pct", "pc2_variance_pct"
]].to_numpy()).all():
    raise ValueError("PCA output contains non-finite values")

panel_b.to_csv(OUTPUT_DIR / "panel_b_pca.csv", index=False)
display(panel_b)
print("Explained variance (%):", pca.explained_variance_ratio_[:2] * 100)

## Panel C — prespecified score trajectories

In [ ]:
target = log_expression.loc[TARGETS].T
target_sd = target.std(axis=0, ddof=0)
if target_sd.le(0).any():
    raise ValueError("A target gene is constant across samples")
target_z = target.subtract(target.mean(axis=0), axis=1).divide(
    target_sd, axis=1
)

scores = pd.DataFrame(index=target_z.index)
scores["Equal4"] = target_z[TARGETS].mean(axis=1)
scores["Core3"] = target_z[["BAK1", "BAX", "BID"]].mean(axis=1)
scores["SLC7A11"] = target_z["SLC7A11"]
scores = scores.reset_index(names="sample_id").merge(
    sample_manifest, on="sample_id", how="left", validate="one_to_one"
)

long_scores = scores.melt(
    id_vars=[
        "sample_id", "geo_accession", "count_column", "condition",
        "replicate", "lps", "carbon_starvation", "torin",
        "duration_hours",
    ],
    value_vars=["Equal4", "Core3", "SLC7A11"],
    var_name="score", value_name="value",
)
panel_c = (
    long_scores.groupby(["score", "condition"], observed=True)["value"]
    .agg(mean="mean", sd="std", n="size")
    .reset_index()
)
panel_c["sem"] = panel_c["sd"] / np.sqrt(panel_c["n"])
panel_c["condition"] = pd.Categorical(
    panel_c["condition"], categories=CONDITIONS, ordered=True
)
panel_c = panel_c.sort_values(["score", "condition"])
if not panel_c["n"].eq(3).all():
    raise ValueError("Each score-condition summary must contain three samples")

panel_c.to_csv(OUTPUT_DIR / "panel_c_score_trajectories.csv", index=False)
long_scores.to_csv(OUTPUT_DIR / "sample_scores.csv", index=False)
display(panel_c)

## Panel D — factorial interaction and Torin-reversal contrasts

In [ ]:
def interaction_result(frame, score_name):
    subset = frame.loc[
        frame["score"].eq(score_name) & frame["torin"].eq(0)
    ].copy()
    design = subset[["lps", "carbon_starvation"]].astype(float).copy()
    design["interaction"] = design["lps"] * design["carbon_starvation"]
    matrix = np.column_stack([
        np.ones(len(design)),
        design[["lps", "carbon_starvation", "interaction"]].to_numpy(),
    ])
    outcome = subset["value"].to_numpy(dtype=float)
    if np.linalg.matrix_rank(matrix) != matrix.shape[1]:
        raise ValueError("The 2×2 factorial design is rank deficient")
    beta = np.linalg.solve(matrix.T @ matrix, matrix.T @ outcome)
    residual = outcome - matrix @ beta
    degrees_freedom = len(outcome) - matrix.shape[1]
    sigma_squared = float(residual @ residual / degrees_freedom)
    covariance = sigma_squared * np.linalg.inv(matrix.T @ matrix)
    standard_error = float(np.sqrt(covariance[3, 3]))
    statistic = float(beta[3] / standard_error)
    p_value = float(2 * stats.t.sf(abs(statistic), degrees_freedom))
    critical = float(stats.t.ppf(0.975, degrees_freedom))
    return {
        "score": score_name,
        "contrast": "LPS-by-CS interaction",
        "n": len(outcome),
        "effect": float(beta[3]),
        "standard_error": standard_error,
        "ci_low": float(beta[3] - critical * standard_error),
        "ci_high": float(beta[3] + critical * standard_error),
        "p_value": p_value,
        "method": "OLS 2x2 interaction; t reference",
    }


def reversal_result(frame, score_name):
    dual = frame.loc[
        frame["score"].eq(score_name)
        & frame["condition"].eq("LPS+CS"), "value"
    ].to_numpy(dtype=float)
    torin = frame.loc[
        frame["score"].eq(score_name)
        & frame["condition"].eq("LPS+CS+Torin"), "value"
    ].to_numpy(dtype=float)
    effect = float(dual.mean() - torin.mean())
    variance = dual.var(ddof=1) / len(dual) + torin.var(ddof=1) / len(torin)
    standard_error = float(np.sqrt(variance))
    denominator = (
        (dual.var(ddof=1) / len(dual)) ** 2 / (len(dual) - 1)
        + (torin.var(ddof=1) / len(torin)) ** 2 / (len(torin) - 1)
    )
    degrees_freedom = float(variance ** 2 / denominator)
    critical = float(stats.t.ppf(0.975, degrees_freedom))
    test = stats.ttest_ind(dual, torin, equal_var=False)
    return {
        "score": score_name,
        "contrast": "Torin reversal",
        "n": len(dual) + len(torin),
        "effect": effect,
        "standard_error": standard_error,
        "ci_low": float(effect - critical * standard_error),
        "ci_high": float(effect + critical * standard_error),
        "p_value": float(test.pvalue),
        "method": "Welch difference in means",
    }


panel_d = pd.DataFrame([
    result
    for score_name in ["Equal4", "Core3", "SLC7A11"]
    for result in [
        interaction_result(long_scores, score_name),
        reversal_result(long_scores, score_name),
    ]
])
if not np.isfinite(panel_d[[
    "effect", "standard_error", "ci_low", "ci_high", "p_value"
]].to_numpy()).all():
    raise ValueError("A Panel D estimate is non-finite")
if not (
    panel_d["ci_low"].le(panel_d["effect"])
    & panel_d["effect"].le(panel_d["ci_high"])
).all():
    raise ValueError("A Panel D estimate falls outside its confidence interval")

panel_d.to_csv(OUTPUT_DIR / "panel_d_contrasts.csv", index=False)
display(panel_d)

## Panel E — seeded expression-matched specificity analysis

In [ ]:
eligible = log_expression.loc[
    log_expression.std(axis=1, ddof=0).gt(0)
].copy()
media_samples = sample_manifest.loc[
    sample_manifest["condition"].eq("Media"), "sample_id"
].tolist()
matching = pd.DataFrame({
    "mean": eligible[media_samples].mean(axis=1),
    "sd": eligible[media_samples].std(axis=1, ddof=0),
})
matching = matching.loc[matching["sd"].gt(0)].copy()
matching_z = matching.subtract(matching.mean()).divide(
    matching.std(ddof=0)
)

pool_rows = []
pools = {}
for target_gene in TARGETS:
    if target_gene not in matching_z.index:
        raise KeyError(
            f"{target_gene} is not eligible for matched-null analysis"
        )
    distances = np.sqrt(
        ((matching_z - matching_z.loc[target_gene]) ** 2).sum(axis=1)
    )
    distances = distances.drop(
        index=[gene for gene in TARGETS if gene in distances.index]
    )
    selected = distances.nsmallest(250)
    pools[target_gene] = selected.index.to_numpy()
    pool_rows.extend({
        "target_gene": target_gene,
        "matched_gene": gene,
        "distance": float(distance),
        "rank": rank,
    } for rank, (gene, distance) in enumerate(selected.items(), start=1))
pd.DataFrame(pool_rows).to_csv(
    OUTPUT_DIR / "matched_null_gene_pools.csv", index=False
)

rng_seed = 20260913
rng = np.random.RandomState(rng_seed)
modules = []
while len(modules) < 10000:
    genes = tuple(rng.choice(pools[target]) for target in TARGETS)
    if len(set(genes)) == len(TARGETS):
        modules.append(genes)

sample_order = sample_manifest["sample_id"].tolist()
gene_z = eligible[sample_order].subtract(
    eligible[sample_order].mean(axis=1), axis=0
).divide(
    eligible[sample_order].std(axis=1, ddof=0), axis=0
)
gene_names = gene_z.index.to_numpy()
gene_position = {gene: index for index, gene in enumerate(gene_names)}
module_indices = np.asarray([
    [gene_position[gene] for gene in module] for module in modules
], dtype=int)
expression_matrix = gene_z.to_numpy(dtype=float)
module_scores = expression_matrix[module_indices].mean(axis=1)

condition = sample_manifest.set_index("sample_id").loc[
    sample_order, "condition"
]

def mean_weight(label):
    mask = condition.eq(label).to_numpy()
    weight = np.zeros(len(mask), dtype=float)
    weight[mask] = 1 / mask.sum()
    return weight


interaction_weight = (
    mean_weight("LPS+CS") - mean_weight("LPS")
    - mean_weight("CS") + mean_weight("Media")
)
reversal_weight = (
    mean_weight("LPS+CS") - mean_weight("LPS+CS+Torin")
)
null_interaction = module_scores @ interaction_weight
null_reversal = module_scores @ reversal_weight

equal4 = scores.set_index("sample_id").loc[
    sample_order, "Equal4"
].to_numpy(dtype=float)
observed_interaction = float(equal4 @ interaction_weight)
observed_reversal = float(equal4 @ reversal_weight)

interaction_exceed = int(np.sum(null_interaction >= observed_interaction))
reversal_exceed = int(np.sum(null_reversal >= observed_reversal))
joint_exceed = int(np.sum(
    (null_interaction >= observed_interaction)
    & (null_reversal >= observed_reversal)
))
n_modules = len(modules)
joint_empirical_p = (joint_exceed + 1) / (n_modules + 1)

panel_e = pd.DataFrame([
    {
        "contrast": "LPS-by-CS interaction",
        "observed_effect": observed_interaction,
        "percentile": 100 * (1 - interaction_exceed / n_modules),
        "empirical_p": (interaction_exceed + 1) / (n_modules + 1),
        "n_modules": n_modules,
        "joint_empirical_p": joint_empirical_p,
        "rng_seed": rng_seed,
    },
    {
        "contrast": "Torin reversal",
        "observed_effect": observed_reversal,
        "percentile": 100 * (1 - reversal_exceed / n_modules),
        "empirical_p": (reversal_exceed + 1) / (n_modules + 1),
        "n_modules": n_modules,
        "joint_empirical_p": joint_empirical_p,
        "rng_seed": rng_seed,
    },
])

null_table = pd.DataFrame({
    "module_id": np.arange(1, n_modules + 1),
    "interaction_effect": null_interaction,
    "reversal_effect": null_reversal,
})
for index, target in enumerate(TARGETS):
    null_table[f"matched_to_{target}"] = [
        module[index] for module in modules
    ]

if not panel_e["percentile"].between(0, 100).all():
    raise ValueError("Matched-null percentile lies outside [0, 100]")
if not panel_e["empirical_p"].between(0, 1).all():
    raise ValueError("Matched-null empirical P lies outside [0, 1]")

panel_e.to_csv(OUTPUT_DIR / "panel_e_specificity.csv", index=False)
null_table.to_csv(OUTPUT_DIR / "matched_null_modules.csv", index=False)
display(panel_e)

## Output audit and plotting handoff

In [ ]:
required_outputs = {
    "normalization_audit.csv": ["sample_id", "size_factor", "condition"],
    "panel_b_pca.csv": [
        "sample_id", "condition", "pc1", "pc2",
        "pc1_variance_pct", "pc2_variance_pct",
    ],
    "panel_c_score_trajectories.csv": [
        "score", "condition", "mean", "sem", "n",
    ],
    "panel_d_contrasts.csv": [
        "score", "contrast", "effect", "ci_low", "ci_high", "p_value",
    ],
    "panel_e_specificity.csv": [
        "contrast", "observed_effect", "percentile", "empirical_p",
        "n_modules", "joint_empirical_p",
    ],
    "sample_scores.csv": ["sample_id", "condition", "score", "value"],
    "matched_null_modules.csv": [
        "module_id", "interaction_effect", "reversal_effect",
    ],
    "matched_null_gene_pools.csv": [
        "target_gene", "matched_gene", "distance", "rank",
    ],
}

audit_rows = []
for filename, columns in required_outputs.items():
    path = OUTPUT_DIR / filename
    if not path.is_file():
        raise FileNotFoundError(path)
    table = pd.read_csv(path)
    missing = sorted(set(columns) - set(table.columns))
    if missing:
        raise KeyError(f"{filename} is missing columns: {missing}")
    audit_rows.append({
        "file": filename,
        "rows": len(table),
        "size_bytes": path.stat().st_size,
        "sha256": sha256_file(path),
    })

output_audit = pd.DataFrame(audit_rows)
output_audit.to_csv(
    OUTPUT_DIR / "figure_7_output_manifest.csv", index=False
)
display(output_audit)
print("\nFigure 7 public-data analysis complete.")
print("Plotting tables saved to:", OUTPUT_DIR)